# Exercice  n°9 : Sélection de modèles

Bienvenue dans cet exercice pratique. Vous allez appliquer les concepts d'évaluation rigoureuse, de validation croisée selon différentes stratégies (incluant par groupes) et de sélection de modèles sur des données de neuroimagerie.

Nous utiliserons le jeu de données **ADHD200**. Pour vous permettre de vous concentrer sur la validation croisée et l'optimisation, la cellule de configuration ci-dessous se chargera d'extraire les features de connectivité fonctionnelle (matrice de corrélation), ainsi que les sites d'acquisition qui serviront de groupes. Elle préparera vos ensembles de données `X_train`, `X_test`, `y_train`, `y_test` et `groups_train`.

Cet exercice totalise **50 points** et couvre :
1. La validation croisée (K-Fold vs Stratified K-Fold)
2. La validation croisée avec groupes (Group K-Fold, Leave-One-Group-Out)
3. L'optimisation exhaustive avec `GridSearchCV`
4. L'optimisation aléatoire avec `RandomizedSearchCV`

**Consignes importantes pour l'évaluation automatique :**

1. **Conformité** : Ne modifiez pas les noms des variables de résultat (ex: `q1_kfold_mean`) donnés dans les lignes commentées.
2. **Déterminisme** : Chaque fois qu'une fonction nécessite un `random_state`, utilisez **EXACTEMENT `random_state=42`**. C'est crucial pour que l'évaluateur automatique puisse valider vos réponses sans ambiguïté.
3. **Exécution** : Assurez-vous que votre code s'exécute sans erreur de haut en bas.

## Section 0 : Configuration et extraction des features

Exécutez cette cellule. Elle télécharge un sous-échantillon des données, extrait les connectomes, isole les sites d'acquisition (groupes) et crée la séparation initiale (Train/Test). **Ne la modifiez pas.**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GroupKFold, LeaveOneGroupOut, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Nombre de sujets pour l'exercice (ajustable par l'autograder)
N_SUBJECTS = 30
data_dir = './nilearn_data'

print("Téléchargement et alignement des données...")
adhd_dataset = datasets.fetch_adhd(n_subjects=N_SUBJECTS, data_dir=data_dir)
pheno_brut = pd.DataFrame(adhd_dataset.phenotypic)

def get_id(path):
    return int(os.path.basename(path).split('_')[0])

func_ids = [get_id(f) for f in adhd_dataset.func]
pheno_ids = set(pheno_brut['Subject'].values)
matched_idx = [i for i, fid in enumerate(func_ids) if fid in pheno_ids]
matched_fids = [func_ids[i] for i in matched_idx]

pheno = pheno_brut.set_index('Subject').loc[matched_fids].reset_index()
func = [adhd_dataset.func[i] for i in matched_idx]
confounds = [adhd_dataset.confounds[i] for i in matched_idx]

basc = datasets.fetch_atlas_basc_multiscale_2015(data_dir=data_dir, resolution=12)
masker = NiftiLabelsMasker(labels_img=basc.maps, standardize='zscore_sample', detrend=True)
cm = ConnectivityMeasure(kind='correlation', vectorize=True, discard_diagonal=True)

print("Extraction des features de connectivité (patientez...)")
all_ts = [masker.fit_transform(f, confounds=c) for f, c in zip(func, confounds)]
X = cm.fit_transform(all_ts)
y = pheno['adhd'].to_numpy(dtype=int)
groups = pheno['site'].to_numpy()  # La variable 'site' servira de groupe d'acquisition

# Création du hold-out test set (80% Train, 20% Test)
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, stratify=y, random_state=42
)
print(f"Terminé ! X_train: {X_train.shape}, X_test: {X_test.shape}")

---
## Partie 1 : Validation croisée standard et stratifiée (10 points)

Nous allons évaluer un modèle `RandomForestClassifier` de base en utilisant deux stratégies de validation croisée différentes sur `X_train` et `y_train`.

### Question 1 : K-fold standard (5 pts)

1. Initialisez un objet `KFold` avec `n_splits=5`, `shuffle=True`, et `random_state=42`.
2. Initialisez un modèle `RandomForestClassifier(random_state=42)`.
3. Utilisez `cross_val_score` pour évaluer le modèle avec ce K-Fold.
4. Stockez la moyenne des scores d'accuracy dans **`q1_kfold_mean`** (float).

In [ ]:
# Réponse 1

# Votre code ici

# q1_kfold_mean = ...

### Question 2 : Stratified K-fold (5 pts)

Le diagnostic TDAH peut être déséquilibré. Une approche stratifiée permet de conserver la proportion des classes dans chaque pli.
1. Initialisez un `StratifiedKFold` avec `n_splits=5`, `shuffle=True`, et `random_state=42`.
2. Utilisez `cross_val_score` avec le même modèle `RandomForestClassifier(random_state=42)`.
3. Stockez la moyenne des scores d'accuracy dans **`q2_stratified_mean`** (float).

In [ ]:
# Réponse 2

# Votre code ici

# q2_stratified_mean = ...

---
## Partie 2 : Validation croisée avec groupes (10 points)

En neuroimagerie, les données proviennent souvent de plusieurs sites d'acquisition. Il est crucial d'évaluer la capacité du modèle à généraliser à de *nouveaux* sites. Pour cela, nous utiliserons `groups_train`.

### Question 3 : Group K-fold (5 pts)

1. Initialisez un `GroupKFold` avec `n_splits=3`.
2. Utilisez `cross_val_score` pour évaluer `RandomForestClassifier(random_state=42)` en lui passant le paramètre `groups=groups_train` en plus de `X_train` et `y_train`.
3. Stockez la moyenne des scores d'accuracy dans **`q3_group_kfold_mean`** (float).

In [ ]:
# Réponse 3

# Votre code ici

# q3_group_kfold_mean = ...

### Question 4 : Leave-one-group-out (5 pts)

Une méthode encore plus stricte consiste à laisser un site d'acquisition entier de côté à chaque itération pour servir de test.
1. Initialisez un objet `LeaveOneGroupOut()`.
2. Utilisez `cross_val_score` avec `RandomForestClassifier(random_state=42)`, toujours en fournissant `groups=groups_train`.
3. Stockez la moyenne des scores d'accuracy dans **`q4_logo_mean`** (float).

In [ ]:
# Réponse 4

# Votre code ici

# q4_logo_mean = ...

---
## Partie 3 : Optimisation exhaustive avec GridSearchCV (15 points)

Nous allons chercher les meilleurs hyperparamètres pour un `SVC` (Support Vector Classifier).

### Question 5 : Configuration du grid search (10 pts)

1. Définissez une grille de paramètres pour un SVC : 
   - `C` : `[0.1, 1, 10]`
   - `kernel` : `['linear', 'rbf']`
2. Initialisez `GridSearchCV` avec un modèle `SVC(random_state=42)`, la grille ci-dessus, et une validation croisée `cv=3`.
3. Entraînez (`fit`) le grid search sur les données d'entraînement (`X_train`, `y_train`).
4. Stockez le meilleur dictionnaire de paramètres dans **`q5_best_params`** (dict) et le meilleur score CV dans **`q5_best_score`** (float).

In [ ]:
# Réponse 5

# Votre code ici

# q5_best_params = ...
# q5_best_score = ...

### Question 6 : Évaluation sur le test set (5 pts)

Le modèle issu du `GridSearchCV` est automatiquement ré-entraîné sur l'ensemble de vos données d'entraînement avec les meilleurs paramètres trouvés.
1. Utilisez l'objet grid search ajusté pour générer des prédictions sur `X_test`.
2. Calculez l'accuracy sur ce jeu de test à l'aide de `accuracy_score`.
3. Stockez ce score final dans **`q6_test_accuracy`** (float).

In [ ]:
# Réponse 6

# Votre code ici

# q6_test_accuracy = ...

---
## Partie 4 : Optimisation aléatoire avec RandomizedSearchCV (15 points)

Lorsque l'espace des hyperparamètres est très vaste (ex: arbres de décision), une recherche aléatoire est souvent plus efficace qu'une recherche exhaustive.

### Question 7 : Recherche aléatoire sur RandomForest (15 pts)

1. Définissez une distribution d'hyperparamètres pour un `RandomForestClassifier` :
   - `n_estimators` : `[10, 50, 100, 200]`
   - `max_depth` : `[None, 5, 10, 20]`
   - `min_samples_split` : `[2, 5, 10]`
2. Initialisez `RandomizedSearchCV` avec un modèle `RandomForestClassifier(random_state=42)`, la distribution ci-dessus, `n_iter=5` (seulement 5 combinaisons testées), `cv=3`, et **obligatoirement `random_state=42`**.
3. Entraînez l'objet sur `X_train`, `y_train`.
4. Stockez les meilleurs paramètres trouvés dans **`q7_best_params`** (dict).
5. Générez les prédictions sur `X_test` et stockez l'accuracy sur le test set dans **`q7_test_accuracy`** (float).

In [ ]:
# Réponse 7

# Votre code ici

# q7_best_params = ...
# q7_test_accuracy = ...